# Image Transformations with R
## Complete Solution Notebook (Expanded R Edition)

Full working R code with alternate implementations, advanced features, rich simulation, and metrics. All outputs are printed.

## Workflow Flowchart
![Workflow Flowchart](image_transform_workflow_flowchart.png)

## 1. Setup & Improved Helper

In [ ]:
library(ggplot2)
library(reshape2)   # for melt() - makes long format easy

show_image <- function(mat, title = "Image", low = "black", high = "white") {
  df <- melt(mat)                    # reshape2::melt is very convenient
  colnames(df) <- c("row", "col", "value")
  p <- ggplot(df, aes(x = col, y = -row, fill = value)) +
    geom_tile(color = "gray30", linewidth = 0.25) +
    scale_fill_gradient(low = low, high = high) +
    coord_fixed() +
    labs(title = title) +
    theme_minimal() +
    theme(
      axis.title = element_blank(),
      axis.text = element_blank(),
      axis.ticks = element_blank(),
      panel.grid = element_blank(),
      plot.title = element_text(hjust = 0.5, face = "bold", size = 11)
    )
  print(p)
  cat(sprintf("[%s] dim=%s, min=%.2f, max=%.2f, mean=%.2f, sd=%.2f\n",
              title, paste(dim(mat), collapse="x"), min(mat), max(mat), mean(mat), sd(mat)))
  invisible(mat)
}

cat("✅ R environment ready with ggplot2 helper.\n")

In [ ]:
heart_img <- matrix(c(
  255,   0,   0, 255,   0,   0, 255,
    0, 127.5, 127.5,   0, 127.5, 127.5,   0,
    0, 127.5, 127.5, 127.5, 127.5, 127.5,   0,
    0, 127.5, 127.5, 127.5, 127.5, 127.5,   0,
  255,   0, 127.5, 127.5, 127.5,   0, 255,
  255, 255,   0, 127.5,   0, 255, 255,
  255, 255, 255,   0, 255, 255, 255
), nrow = 7, byrow = TRUE)

show_image(heart_img, "Original Heart Image")

## 2. Matrix Inspection (R style)

In [ ]:
cat("=== MATRIX INSPECTION ===\n")
cat("dim     :", dim(heart_img), "\n")
cat("class   :", class(heart_img), "\n")
cat("Center (4,4):", heart_img[4,4], "\n")
cat("Top row :", heart_img[1,], "\n")
cat("Left col:", heart_img[,1], "\n")

## 3. Basic Transformations — Multiple Alternate Methods

### 3.1 Color Inversion — Three Ways

In [ ]:
cat("=== INVERSION ALTERNATIVES ===\n")
inv1 <- 255 - heart_img
show_image(inv1, "Invert: 255 - img (recommended)")

inv2 <- max(heart_img) - heart_img
show_image(inv2, "Invert: max() - img")

# For integer matrices
inv3 <- 255L - as.integer(heart_img)
show_image(inv3, "Invert: integer subtraction")

### 3.2 Geometric Transforms

In [ ]:
cat("=== GEOMETRIC TRANSFORMS ===\n")
show_image(heart_img[, ncol(heart_img):1], "Horizontal Flip")
show_image(heart_img[nrow(heart_img):1, ], "Vertical Flip")
show_image(t(heart_img), "Transpose (t())")
show_image(t(heart_img[nrow(heart_img):1, ]), "90° Clockwise Rotation")

## 4. Brightness, Contrast & Noise (Data Augmentation)

In [ ]:
cat("=== BRIGHTNESS & NOISE ===\n")
for (factor in c(0.6, 1.0, 1.7)) {
  b <- pmin(pmax(heart_img * factor, 0), 255)
  show_image(b, paste0("Brightness x", factor))
}

set.seed(42)
for (sd_val in c(10, 40, 85)) {
  noisy <- pmin(pmax(heart_img + matrix(rnorm(49, 0, sd_val), 7), 0), 255)
  show_image(noisy, paste0("Gaussian Noise sd=", sd_val))
}

## 5. Filtering — Custom Gaussian Blur + Edge Emphasis

In [ ]:
# Reusable Gaussian blur (improved padding)
gaussian_blur <- function(img, sigma = 1) {
  k <- ceiling(3 * sigma)
  x <- seq(-k, k)
  kern <- outer(x, x, function(a,b) dnorm(a,0,sigma) * dnorm(b,0,sigma))
  kern <- kern / sum(kern)
  padded <- rbind(
    matrix(rep(img[1,], k), k, byrow=TRUE),
    img,
    matrix(rep(img[nrow(img),], k), k, byrow=TRUE)
  )
  padded <- cbind(
    matrix(rep(padded[,1], k), ncol=k),
    padded,
    matrix(rep(padded[,ncol(padded)], k), ncol=k)
  )
  out <- img * 0
  for (i in seq_len(nrow(img))) {
    for (j in seq_len(ncol(img))) {
      region <- padded[i:(i+2*k), j:(j+2*k)]
      out[i,j] <- sum(region * kern)
    }
  }
  out
}

for (s in c(0.5, 1.2, 2.0)) {
  bl <- gaussian_blur(heart_img, sigma = s)
  show_image(bl, paste0("Gaussian Blur sigma=", s))
}

# Simple edge emphasis
edge_k <- matrix(c(-1,-1,-1, -1,8,-1, -1,-1,-1), 3, 3)
edges <- heart_img * 0
for (i in 2:6) for (j in 2:6) edges[i,j] <- sum(heart_img[(i-1):(i+1),(j-1):(j+1)] * edge_k)
show_image(pmax(edges, 0), "Simple Edge Emphasis")

## 6. Linear Algebra — Image Recovery

In [ ]:
cat("=== LINEAR ALGEBRA RECOVERY ===\n")
set.seed(123)
A <- matrix(sample(20:230, 49, replace=TRUE), nrow=7)
show_image(A, "Random mixing matrix A")

X <- solve(A, heart_img)          # R's solve() is excellent
show_image(X, "Solved X")

rec <- A %*% X
show_image(rec, "Reconstructed = A %*% X")
cat("Frobenius error:", sqrt(sum((rec - heart_img)^2)), "\n")

# Noisy case with least squares (MASS::ginv or qr)
A_noisy <- A + matrix(rnorm(49, 0, 8), 7)
X_ls <- qr.solve(A_noisy, heart_img)   # or MASS::ginv
rec_ls <- A_noisy %*% X_ls
cat("Least-squares error on noisy A:", sqrt(sum((rec_ls - heart_img)^2)), "\n")

## 7. Full Simulation Dashboard (R)

In [ ]:
cat("\n", strrep("=", 55), "\n")
cat("R SIMULATION DASHBOARD — Change values and re-run\n")
cat(strrep("=", 55), "\n\n")


In [ ]:
# === USER CONTROLS ===
brightness <- 1.35
noise_sd   <- 18
blur_sigma <- 0.7
do_invert  <- FALSE
flip_mode  <- "none"     # "h", "v", "none"

# === PIPELINE ===
img <- heart_img
img <- pmin(pmax(img * brightness, 0), 255)
if (noise_sd > 0) {
  set.seed(42)
  img <- img + matrix(rnorm(length(img), 0, noise_sd), nrow = 7)
  img <- pmin(pmax(img, 0), 255)
}
if (blur_sigma > 0) img <- gaussian_blur(img, sigma = blur_sigma)
if (do_invert) img <- 255 - img
if (flip_mode == "h") img <- img[, ncol(img):1]
if (flip_mode == "v") img <- img[nrow(img):1, ]

# === METRICS ===
mae  <- mean(abs(img - heart_img))
mse  <- mean((img - heart_img)^2)
rmse <- sqrt(mse)
psnr <- if (rmse > 0) 20 * log10(255 / rmse) else Inf
cat(sprintf("brightness=%.2f | noise_sd=%.1f | blur=%.2f | invert=%s | flip=%s\n",
            brightness, noise_sd, blur_sigma, do_invert, flip_mode))
cat(sprintf("MAE = %.2f   RMSE = %.2f   PSNR = %.2f dB\n", mae, rmse, psnr))

# Visual comparison
par(mfrow = c(2, 2))
show_image(heart_img, "Original")
show_image(img, "Transformed")
show_image(abs(img - heart_img), "Absolute Difference", low = "white", high = "red")
hist(abs(img - heart_img), breaks = 25, col = "#E91E63", main = "Pixel Diff Distribution", xlab = "|diff|")
par(mfrow = c(1, 1))

## 8. Practice Challenges — Sample Solutions

In [ ]:
# Custom shape: plus sign
plus <- matrix(0, 9, 9)
plus[5, 3:7] <- 220
plus[3:7, 5] <- 220
plus[c(2,8), c(4,6)] <- 160
show_image(plus, "Custom Plus Shape")

# Chain transforms
chained <- pmin(pmax(plus * 0.75 + matrix(rnorm(81,0,12),9),0),255)
chained <- gaussian_blur(chained, sigma = 0.8)
show_image(chained, "Brightness + Noise + Blur chain")

## Key Insights & Audience Considerations (R)

R's matrix algebra (`%*%`, `solve()`, `t()`) is extremely elegant for this type of work. The same principles of **audience awareness** apply: show executives the before/after visuals + one business sentence; show data scientists the code and error metrics.

---
## Appendix: R Quick Reference

| Operation       | R Code                              | Notes                          |
|-----------------|-------------------------------------|--------------------------------|
| Invert          | `255 - img` or `max(img) - img`    | Works on matrices              |
| Horizontal flip | `img[, ncol(img):1]`                | -                              |
| Transpose       | `t(img)`                            | -                              |
| Brightness      | `pmin(pmax(img * f, 0), 255)`       | Always clip                    |
| Gaussian noise  | `img + matrix(rnorm(n), nrow)`      | -                              |
| Solve system    | `solve(A, B)` or `qr.solve(A, B)`   | Excellent in R                 |
| Metrics         | `mean(abs(diff))`, `sqrt(mean((diff)^2))` | Easy to compute         |

**You have completed the full R solution notebook!**

Next steps: Try the `imager` package for production-quality image processing, or move to `torch` for deep learning transforms in R.